# Simple Reflex Agent Project

## Applied Machine Intelligence

### Overview

This project develops a simple reflex agent that operates in a simulated two-room vacuum environment. The agent senses its current location and whether that location is clean or dirty. It then selects an action using predefined condition–action rules.

The agent does not store previous states, learn from experience, or predict future conditions. Its decisions are based only on the current percept received from the environment.

## 1. Environment Design

The simulated environment is based on the classic two-room vacuum world. It contains two locations: **Room A** and **Room B**. Each room can be in one of two states: **Clean** or **Dirty**.

The agent can perform the following actions:

* **Clean**: Removes dirt from the current room.
* **Move Right**: Moves the agent from Room A to Room B.
* **Move Left**: Moves the agent from Room B to Room A.
* **Stop**: Ends the simulation when both rooms are clean.

At each step, the agent receives a percept containing its current location, the condition of the current room, and whether all rooms are clean. The agent selects an action using predefined condition–action rules and does not use memory or learning.


## 2. Condition–Action Rules

The agent receives a current percept containing:

* The agent’s current location
* The condition of the current room
* Whether all rooms are clean

The agent applies the following condition–action rules:

| Current condition                        | Agent action |
| ---------------------------------------- | ------------ |
| All rooms are clean                      | Stop         |
| Current room is dirty                    | Clean        |
| Agent is in Room A and the room is clean | Move Right   |
| Agent is in Room B and the room is clean | Move Left    |

These rules are evaluated directly from the current percept. The agent does not store previous actions, maintain a history, learn from earlier test runs, or predict future conditions.


### Explanation of the Environment Code

#### `VacuumEnvironment` Class

The `VacuumEnvironment` class represents the simulated two-room vacuum world. It stores the condition of each room and the agent’s current location.

#### `__init__()` Method

The constructor initializes the environment using three values:

- The starting condition of Room A
- The starting condition of Room B
- The agent’s starting location

The room conditions are stored in a Python dictionary named `rooms`, while the agent’s current position is stored in `agent_location`.

#### `get_percept()` Method

This method allows the agent to sense the current environment. It returns a percept containing:

- The agent’s current location
- The condition of the room where the agent is located
- Whether both rooms are clean

This percept provides the information that the simple reflex agent uses to select its next action.

#### `execute_action()` Method

This method updates the environment after the agent selects an action.

- `Clean` changes the current room’s condition to clean.
- `Move Right` moves the agent to Room B.
- `Move Left` moves the agent to Room A.
- `Stop` leaves the environment unchanged and ends the simulation.

An error is raised if an unsupported action is provided.

#### `display_state()` Method

This method creates a readable description of the current environment. It displays the agent’s location and the current condition of both rooms. This will help us track the agent’s behavior during each test run.


In [1]:
class VacuumEnvironment:
    """
    Simulates a two-room vacuum environment.

    Each room can be either 'Clean' or 'Dirty'.
    The agent can move between Room A and Room B and clean the current room.
    """

    def __init__(self, room_a_state, room_b_state, starting_location):
        self.rooms = {
            "A": room_a_state,
            "B": room_b_state
        }
        self.agent_location = starting_location

    def get_percept(self):
        """
        Returns the agent's current percept.

        The percept includes:
        - current location
        - condition of the current room
        - whether all rooms are clean
        """
        current_condition = self.rooms[self.agent_location]
        all_clean = all(state == "Clean" for state in self.rooms.values())

        return {
            "location": self.agent_location,
            "condition": current_condition,
            "all_clean": all_clean
        }

    def execute_action(self, action):
        """Updates the environment based on the agent's selected action."""

        if action == "Clean":
            self.rooms[self.agent_location] = "Clean"

        elif action == "Move Right":
            self.agent_location = "B"

        elif action == "Move Left":
            self.agent_location = "A"

        elif action == "Stop":
            pass

        else:
            raise ValueError(f"Unknown action: {action}")

    def display_state(self):
        """Displays the current state of the environment."""
        return (
            f"Location: Room {self.agent_location}, "
            f"Room A: {self.rooms['A']}, "
            f"Room B: {self.rooms['B']}"
        )

## 3. Simple Reflex Agent Design

The simple reflex agent receives the current percept from the environment and selects an action using fixed condition–action rules.

The agent does not change the environment directly. Instead, it returns an action, and the environment executes that action.

The decision process follows this order:

1. If all rooms are clean, select **Stop**.
2. If the current room is dirty, select **Clean**.
3. If the current room is clean and the agent is in Room A, select **Move Right**.
4. If the current room is clean and the agent is in Room B, select **Move Left**.

The function uses only the percept provided during the current step. It does not store previous percepts, actions, or test results.

In [2]:
def simple_reflex_agent(percept):
    """
    Selects an action based only on the current percept.

    Parameters:
        percept (dict): Contains the current location, current room
                        condition, and whether all rooms are clean.

    Returns:
        str: The action selected by the agent.
    """

    location = percept["location"]
    condition = percept["condition"]
    all_clean = percept["all_clean"]

    # Rule 1: Stop when the entire environment is clean.
    if all_clean:
        return "Stop"

    # Rule 2: Clean the current room when it is dirty.
    if condition == "Dirty":
        return "Clean"

    # Rule 3: Move from Room A to Room B when Room A is clean.
    if location == "A":
        return "Move Right"

    # Rule 4: Move from Room B to Room A when Room B is clean.
    if location == "B":
        return "Move Left"

    # Raise an error if the percept contains an unsupported location.
    raise ValueError(f"Unknown location: {location}")

## 4. Simulation Runner

The simulation runner connects the environment and the simple reflex agent.

During each step, the runner performs the following tasks:

1. Requests the current percept from the environment.
2. Sends the percept to the simple reflex agent.
3. Records the current state and selected action.
4. Executes the action in the environment.
5. Stops the simulation when the agent selects **Stop**.

A maximum number of steps is included as a safety measure to prevent an infinite loop if the rules or environment are configured incorrectly.

In [3]:
def run_simulation(
    room_a_state,
    room_b_state,
    starting_location,
    test_name,
    max_steps=10
):
    """
    Runs one vacuum-world test scenario.

    Parameters:
        room_a_state (str): Initial condition of Room A.
        room_b_state (str): Initial condition of Room B.
        starting_location (str): Agent's starting room, 'A' or 'B'.
        test_name (str): Descriptive name of the test.
        max_steps (int): Maximum number of simulation steps.

    Returns:
        list: A record of the agent's actions during the simulation.
    """

    environment = VacuumEnvironment(
        room_a_state=room_a_state,
        room_b_state=room_b_state,
        starting_location=starting_location
    )

    action_history = []

    print("=" * 70)
    print(test_name)
    print("=" * 70)
    print(f"Initial State: {environment.display_state()}\n")

    for step in range(1, max_steps + 1):
        # The agent senses only the current percept.
        percept = environment.get_percept()

        # The agent selects an action using condition-action rules.
        action = simple_reflex_agent(percept)

        # Record the current step for later review.
        step_record = {
            "step": step,
            "location": percept["location"],
            "condition": percept["condition"],
            "all_clean": percept["all_clean"],
            "action": action
        }
        action_history.append(step_record)

        print(
            f"Step {step}: "
            f"Percept = {percept} | "
            f"Action = {action}"
        )

        # Stop when the agent reports that all rooms are clean.
        if action == "Stop":
            break

        # Update the environment based on the selected action.
        environment.execute_action(action)

    else:
        print("\nMaximum number of steps reached.")

    print(f"\nFinal State: {environment.display_state()}")
    print(f"Total Steps: {len(action_history)}")
    print()

    return action_history

## 5. Test Scenarios

The agent is tested using different starting conditions to verify that the condition–action rules work correctly.

The following scenarios are included:

1. **Test 1:** Both rooms are dirty, and the agent starts in Room A.
2. **Test 2:** Room A is clean, Room B is dirty, and the agent starts in Room A.
3. **Test 3:** Both rooms are clean, and the agent starts in Room B.
4. **Test 4:** Room A is dirty, Room B is clean, and the agent starts in Room B.

These scenarios test cleaning, movement between rooms, starting from different locations, and stopping when the environment is already clean.

In [4]:
# Test 1: Both rooms are dirty, starting in Room A
test_1_history = run_simulation(
    room_a_state="Dirty",
    room_b_state="Dirty",
    starting_location="A",
    test_name="Test 1: Both Rooms Dirty - Start in Room A"
)

Test 1: Both Rooms Dirty - Start in Room A
Initial State: Location: Room A, Room A: Dirty, Room B: Dirty

Step 1: Percept = {'location': 'A', 'condition': 'Dirty', 'all_clean': False} | Action = Clean
Step 2: Percept = {'location': 'A', 'condition': 'Clean', 'all_clean': False} | Action = Move Right
Step 3: Percept = {'location': 'B', 'condition': 'Dirty', 'all_clean': False} | Action = Clean
Step 4: Percept = {'location': 'B', 'condition': 'Clean', 'all_clean': True} | Action = Stop

Final State: Location: Room B, Room A: Clean, Room B: Clean
Total Steps: 4



In [5]:
# Test 2: Room A is clean and Room B is dirty, starting in Room A
test_2_history = run_simulation(
    room_a_state="Clean",
    room_b_state="Dirty",
    starting_location="A",
    test_name="Test 2: Room A Clean, Room B Dirty - Start in Room A"
)

Test 2: Room A Clean, Room B Dirty - Start in Room A
Initial State: Location: Room A, Room A: Clean, Room B: Dirty

Step 1: Percept = {'location': 'A', 'condition': 'Clean', 'all_clean': False} | Action = Move Right
Step 2: Percept = {'location': 'B', 'condition': 'Dirty', 'all_clean': False} | Action = Clean
Step 3: Percept = {'location': 'B', 'condition': 'Clean', 'all_clean': True} | Action = Stop

Final State: Location: Room B, Room A: Clean, Room B: Clean
Total Steps: 3



In [6]:
# Test 3: Both rooms are clean, starting in Room B
test_3_history = run_simulation(
    room_a_state="Clean",
    room_b_state="Clean",
    starting_location="B",
    test_name="Test 3: Both Rooms Clean - Start in Room B"
)

Test 3: Both Rooms Clean - Start in Room B
Initial State: Location: Room B, Room A: Clean, Room B: Clean

Step 1: Percept = {'location': 'B', 'condition': 'Clean', 'all_clean': True} | Action = Stop

Final State: Location: Room B, Room A: Clean, Room B: Clean
Total Steps: 1



In [7]:
# Test 4: Room A is dirty and Room B is clean, starting in Room B
test_4_history = run_simulation(
    room_a_state="Dirty",
    room_b_state="Clean",
    starting_location="B",
    test_name="Test 4: Room A Dirty, Room B Clean - Start in Room B"
)

Test 4: Room A Dirty, Room B Clean - Start in Room B
Initial State: Location: Room B, Room A: Dirty, Room B: Clean

Step 1: Percept = {'location': 'B', 'condition': 'Clean', 'all_clean': False} | Action = Move Left
Step 2: Percept = {'location': 'A', 'condition': 'Dirty', 'all_clean': False} | Action = Clean
Step 3: Percept = {'location': 'A', 'condition': 'Clean', 'all_clean': True} | Action = Stop

Final State: Location: Room A, Room A: Clean, Room B: Clean
Total Steps: 3



## 6. Test Results and Observations

All four test scenarios produced the expected results.

| Test   | Initial Condition          | Starting Location | Actions Taken                     | Final Outcome             |
| ------ | -------------------------- | ----------------- | --------------------------------- | ------------------------- |
| Test 1 | Room A dirty, Room B dirty | Room A            | Clean → Move Right → Clean → Stop | Both rooms clean          |
| Test 2 | Room A clean, Room B dirty | Room A            | Move Right → Clean → Stop         | Both rooms clean          |
| Test 3 | Room A clean, Room B clean | Room B            | Stop                              | Both rooms remained clean |
| Test 4 | Room A dirty, Room B clean | Room B            | Move Left → Clean → Stop          | Both rooms clean          |

The results show that the agent correctly applies its condition–action rules. When the current room is dirty, the agent cleans it. When the current room is clean but another room is dirty, the agent moves to the other location. When both rooms are clean, the agent stops.

The agent also behaves correctly when starting in either Room A or Room B. However, its behavior depends entirely on the current percept and predefined rules. It does not remember previous actions or learn from the test scenarios.


## 7. Performance and Limitations

The simple reflex agent performed successfully in all four test scenarios. It cleaned dirty rooms, moved to the appropriate location, and stopped after the environment became clean. Its behavior was predictable because each action was selected from a fixed set of condition–action rules.

One advantage of this design is its simplicity. The agent requires little computation and responds immediately to the current percept. This makes it suitable for small, fully observable environments where every relevant condition can be sensed directly.

However, the agent has several limitations:

- It does not remember previously visited locations or actions.
- It cannot learn from experience or improve its behavior.
- It depends on the environment to report whether all rooms are clean.
- It may repeatedly move between rooms if the environment changes continuously.
- Its rules are designed specifically for a two-room environment.
- It cannot plan an efficient route for a larger environment.

A more advanced agent could maintain an internal model of the environment, remember which rooms were previously cleaned, support additional rooms, and calculate a more efficient cleaning path.

## 8. Reflection and Possible Improvements

This project helped demonstrate how a simple reflex agent makes decisions using only the current percept. The agent does not need memory, learning, or planning because each environmental condition is directly connected to a specific action.

The most important part of the design was creating clear condition–action rules. The order of these rules also mattered. For example, the agent first checks whether all rooms are clean before deciding to move. Without this check, the agent could continue moving between rooms even after cleaning was complete.

The project also showed that simple reflex agents work well in small and fully observable environments. However, they become less effective when the environment is larger, partially observable, or constantly changing.

The agent could be improved by:

- Adding more rooms and movement options
- Tracking previously visited locations
- Storing an internal representation of room conditions
- Measuring the number of movements and cleaning actions
- Assigning costs to actions
- Selecting actions that minimize time or energy
- Allowing the environment to become dirty again during the simulation

These improvements would change the design from a simple reflex agent toward a model-based, goal-based, or utility-based agent.